## 1. Importing libraries

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import zscore, skew
import seaborn as sns
import matplotlib.pyplot as plt
import sys
sys.path.append("../scripts")
from outlier_utils import print_outliers

## 2. Data Loading 

In [3]:
df = pd.read_csv("../data/kenya.csv")
df.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64


## 3. Adding column & Date Parsing

In [4]:
# Add country column
df["Country"] = "Kenya"

# Convert YEAR + DOY to datetime
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract month
df["Month"] = df["Date"].dt.month

df.head()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,Month
0,2015,1,19.56,28.99,12.09,16.90,0.00,45.32,3.12,4.76,83.68,6.88,Kenya,2015-01-01,1
1,2015,2,19.63,29.77,11.04,18.73,0.00,38.76,3.23,4.35,83.67,5.85,Kenya,2015-01-02,1
2,2015,3,20.40,30.57,11.71,18.86,0.00,41.75,3.46,4.68,83.69,6.65,Kenya,2015-01-03,1
3,2015,4,21.33,31.20,13.02,18.18,3.49,51.87,2.29,4.00,83.62,8.60,Kenya,2015-01-04,1
4,2015,5,20.41,29.52,12.38,17.14,1.79,48.04,1.77,4.05,83.54,7.64,Kenya,2015-01-05,1


- The dataset encodes dates using Year and Day-of-Year (DOY), which was converted into a proper datetime format to enable time-series analysis and seasonal interpretations.

## 4. Summary Statistics & Missing Values

In [5]:
# Replace sentinel values
df.replace(-999, np.nan, inplace=True)

# Duplicates
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

df = df.drop_duplicates()

Duplicate rows: 0


- No duplicates were found in any of the columns, which means every entry is unique

In [6]:
# Summary statistics
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Date,Month
count,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108.000000,4108,4108.000000
mean,2020.131451,180.121227,20.427600,27.838717,14.673169,13.165548,1.468162,65.845355,3.061765,4.375241,83.724335,11.052539,2020-08-15 12:00:00,6.423564
min,2015.000000,1.000000,15.260000,18.880000,8.970000,4.110000,0.000000,28.420000,0.610000,1.160000,83.310000,4.780000,2015-01-01 00:00:00,1.000000
25%,2017.000000,86.000000,19.460000,26.297500,13.700000,11.467500,0.100000,58.677500,2.420000,3.670000,83.630000,9.880000,2017-10-23 18:00:00,3.000000
50%,2020.000000,179.000000,20.360000,27.875000,14.750000,13.260000,0.380000,66.220000,3.140000,4.430000,83.720000,11.005000,2020-08-15 12:00:00,6.000000
75%,2023.000000,272.000000,21.400000,29.520000,15.750000,15.050000,1.360000,73.280000,3.720000,5.090000,83.810000,12.350000,2023-06-08 06:00:00,9.000000
max,2026.000000,366.000000,25.400000,34.270000,18.750000,20.120000,51.650000,91.070000,5.280000,7.590000,84.170000,15.040000,2026-03-31 00:00:00,12.000000
std,3.248907,106.294767,1.440824,2.358770,1.415691,2.605174,3.180228,9.934196,0.853218,0.992156,0.126391,1.607151,NaN,3.477046


### Brief interpretation of the statistics
### 1. **General Observations**
- **Time span**: 4,108 days (~11.25 years), from 2015 through most of 2026.
- **DOY (Day of Year)** ranges from 1 to 366, confirming leap years are included (e.g., 2020, 2024).
---

### 2. **Temperature Variables (°C)**

| Variable | Mean | Min | Max | Interpretation |
|----------|------|-----|-----|----------------|
| **T2M** | 20.4 | 15.3 | 25.4 | Mild, semi-tropical or temperate climate; no freezing temperatures. |
| **T2M_MAX** | 27.8 | 18.9 | 34.3 | Hot days occur, but rare extremes above 34°C. |
| **T2M_MIN** | 14.7 | 9.0 | 18.8 | Nights are cool-to-mild; never freezing. |
| **T2M_RANGE** | 13.2 | 4.1 | 20.1 | Moderate diurnal variation; largest swings in dry/clear conditions. |

**Key insight**: The site has warm days and mild nights, with a mean diurnal range of ~13°C, typical of a **subtropical or semi-arid inland** location.

---

### 3. **Precipitation (PRECTOTCORR – mm/day)**
- **Mean**: 1.47 mm/day → ~536 mm/year (semi-arid to sub-humid).
- **Median**: 0.38 mm/day → most days have little or no rain.
- **Max**: 51.65 mm/day → rare but heavy rain events.
- **75th percentile**: 1.36 mm/day → even on rainier days, intensity is low.

**Interpretation**: Strongly **right-skewed** – dry conditions dominate, but occasional heavy downpours occur.

---

### 4. **Humidity & Moisture**

| Variable | Mean | Min–Max | Notes |
|----------|------|---------|-------|
| **RH2M (%)** | 65.8 | 28.4 – 91.1 | Moderate relative humidity; not persistently dry nor humid. |
| **QV2M (g/kg)** | 11.05 | 4.78 – 15.04 | Moderate specific humidity; corresponds to ~11 g water vapor per kg of air. |

**Interpretation**: Air is moderately moist; RH drops to ~28% on dry days, but specific humidity never extremely low, suggesting some moisture source.

---

### 5. **Wind Speed (m/s)**

| Variable | Mean | 75th %ile | Max | Interpretation |
|----------|------|-----------|-----|----------------|
| **WS2M** | 3.06 | 3.72 | 5.28 | Light to moderate mean winds. |
| **WS2M_MAX** | 4.38 | 5.09 | 7.59 | Gusts occasionally strong. |

**Interpretation**: Breezy but not extremely windy; max gusts ~7.6 m/s (~27 km/h).

---

### 6. **Atmospheric Pressure (PS – kPa)**
- **Mean**: 83.72 kPa  
- **Range**: 83.31 – 84.17 kPa  

Pressure is **lower than sea-level standard (~101.3 kPa)**, indicating the station is at **elevation (~1,600–1,800 meters)**.  
- Formula approximate: `Elevation (m) ≈ (101.3 - PS) * 100 / 1.2` → around **1,450–1,650 m** elevation.

---

## **Overall Climate Summary**
- **Climate type**: Warm temperate / subtropical highland or semi-arid.
- **Elevation**: ~1,500 m above sea level (inferred from PS).
- **Rainfall pattern**: Mostly dry; heavy rain infrequent.
- **Temperature**: No frost; comfortable mean annual temp ~20°C.
- **Diurnal range**: Moderate to large (~13°C).
- **Humidity**: Moderate; specific humidity relatively stable.

## 5. Missing values

In [7]:
# Missing values
missing = df.isna().sum()
missing_percent = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
})

missing_df

,Missing Count,Missing %
YEAR,0,0.0
DOY,0,0.0
T2M,0,0.0
T2M_MAX,0,0.0
T2M_MIN,0,0.0
T2M_RANGE,0,0.0
PRECTOTCORR,0,0.0
RH2M,0,0.0
WS2M,0,0.0
WS2M_MAX,0,0.0


- there are no null values, which makes our data interpretations and insights we draw from it more accurate since we dont miss anything

## 6. Outlier Detection

In [8]:
cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]

# Calculate z-scores
z_scores = np.abs(zscore(df[cols], nan_policy='omit'))

# Create named DataFrame for easier access
zscore_df = pd.DataFrame(
    z_scores, 
    columns=cols, 
    index=df.index
)

# FLAG the outliers (create a new column)
df['is_outlier'] = (zscore_df > 3).any(axis=1)  # ← THIS IS THE FLAG

# Report the counts
outliers = (z_scores > 3)
outlier_rows = df['is_outlier'].sum()
print("Outlier rows:", outlier_rows)
print(f"Percentage: {df['is_outlier'].mean()*100:.2f}%")

Outlier rows: 121
Percentage: 2.95%


- We have 121 rows with outlier values at one or more column, so now we check if they are erroneous values or seasonal extremities to make a decision whether to keep them or remove them.

### Check outliers by column

In [9]:
print_outliers(zscore_df, cols)

T2M: 8 outliers
T2M_MAX: 3 outliers
T2M_MIN: 9 outliers
PRECTOTCORR: 92 outliers
RH2M: 6 outliers
WS2M: 0 outliers
WS2M_MAX: 6 outliers


### Evaluating outliers

In [10]:
# 1. Physical possibility checks
impossible = df[
    (df["T2M_MAX"] < df["T2M_MIN"]) |  # Max < Min
    (df["RH2M"] > 100) |                # RH > 100%
    (df["RH2M"] < 0) |                  # RH < 0%
    (df["PRECTOTCORR"] < 0)             # Negative rain
]
print(f"Physically impossible rows: {len(impossible)}")

Physically impossible rows: 0


- We have no physically impossible rows, all values are within plausible ranges.

In [12]:
# 2. Check if outliers are seasonal
rain_outliers = df[zscore_df["PRECTOTCORR"] > 3] 
print("Precipitation outliers by month:")
print(rain_outliers["Month"].value_counts().sort_index())

Precipitation outliers by month:
Month
1      8
2      8
3     17
4     28
5      7
8      2
10     5
11     7
12    10
Name: count, dtype: int64


**Total daily precipitation (PRECTOTCORR)**

**Decision**: We will keep the precipitation outliers.

**Reasons:**
1. **Clear seasonal pattern confirms these are real weather events**
- **Peak in April (28 outliers)** and **March (17 outliers)** - likely the rainy season peak
- **December-March (8-17 outliers)** - wet season months
- **August only has 2 outliers** - dry season month with rare events
- This matches real climatology where heavy rain occurs in specific seasons

2. **The distribution makes meteorological sense**
- Outliers concentrate in months 3-4 (spring/early wet season)
- Very few in month 8 (dry season)
- The pattern suggests **convective storms**, not data errors

3. **The outlier percentage is reasonable**
- 92 / 4108 total days = **~2.2% of all days**
- This is a realistic frequency for heavy precipitation events

**What these outliers likely represent:**
- **April peak (28 events)** = End of dry season / start of wet season thunderstorms
- **March (17 events)** = Early season convection
- **These are scientifically important** for understanding extreme precipitation timing

In [18]:
min_temp_outliers = df[zscore_df["T2M_MIN"] > 3]
print("Minimun temprature outliers by month:")
print(min_temp_outliers["Month"].value_counts().sort_index())

temp_outliers = df[zscore_df["T2M"] > 3]
print("Temprature outliers by month:")
print(temp_outliers["Month"].value_counts().sort_index())

max_temp_outliers = df[zscore_df["T2M_MAX"] > 3]
print("Maximum Temprature outliers by month:")
print(temp_outliers["Month"].value_counts().sort_index())

Minimun temprature outliers by month:
Month
6    1
7    7
8    1
Name: count, dtype: int64
Temprature outliers by month:
Month
3    6
5    1
7    1
Name: count, dtype: int64
Maximum Temprature outliers by month:
Month
3    6
5    1
7    1
Name: count, dtype: int64


**Decision**: We will keep the temperature outliers.

**Minimum Temperature Outliers**
**Distribution:** June (1), July (7), August (1) = **9 total outliers**

**Reasons:**
These represent **unusually warm minimum temperatures** (since Z-score > 3 means higher than normal):

- **Peak in July (7 outliers)** - Midsummer nights staying unusually warm
- **Sporadic in June/August** - Shoulder months of summer
- **Only 9 days total (0.22% of data)** - Very rare events

**Mean and Max Temperature Outliers**

**Distribution:** March (6), May (1), July (1) = **8 total outliers**

**Reasons:**
These represent **unusually warm average temperatures**:

- **Peak in March (6 outliers)** - Early spring heat waves
- **Single events in May/July** - Occasional summer extremes. This pattern is valuable for climate studies (early heat waves can be more damaging to agriculture)
- **Only 8 days total (0.19% of data)** - Extremely rare

In [14]:
relative_humidity_outliers = df[zscore_df["RH2M"] > 3]
print("Relative humidity outliers by month:")
print(relative_humidity_outliers["Month"].value_counts().sort_index())

Relative humidity outliers by month:
Month
1    5
2    1
Name: count, dtype: int64


**Decision**: We will keep the humidity outliers.

**Relative Humidity Outliers (RH2M)**
**Distribution:** January(5), February(1)
   > RH values of 28-36% in January-February are characteristic of Kenya's dry season and were retained as valid observations. No data quality issues were identified.




In [17]:
max_daily_wind_speed_outliers = df[zscore_df["WS2M_MAX"] > 3]
print("Max daily wind speed outliers outliers by month:")
print(max_daily_wind_speed_outliers["Month"].value_counts().sort_index())

Max daily wind speed outliers outliers by month:
Month
1     2
2     1
4     1
5     1
12    1
Name: count, dtype: int64


**Decision**: We will keep the wind speed outliers.

**Maximum Daily Wind Speed Outliers (WS2M_MAX)**

**Distribution:**
- **January**: 2 outliers
- **February**: 1 outlier  
- **April**: 1 outlier
- **May**: 1 outlier
- **December**: 1 outlier
- **Total**: 6 outliers (0.15% of data)

**Reasons**
> Maximum daily wind speed outliers (<0.2% of observations) occurred primarily during the North-East Monsoon season (December-February) and pre-rain period (April-May). Values ranging are consistent with known wind patterns in Kenya and were retained as valid extreme events.

## 7. Export Clean Data

In [38]:
df.to_csv(f"../data/kenya_clean.csv", index=False)